---
toc: false
title: Why Causality — the Prediction-Decision Gap
date: "10/30/2026"
description: Correlation is not causation, except when it is.
categories: causal_inference
image: /enso-thumbnail.jpg
image-alt: The enso circle
draft: true
---


## Why causal inference 

### The Purpose of Data Science

Every once in a while my mom asks me if I can re-explain to her what data scientists do. It's a great question really—we in the field should ask ourselves more often. In a single sentence I'd say: the purpose of data science is to create value from data.  And what's valuable to the organizations we serve?  Well an organization has goals, and in order to achieve those goals it needs to make decisions about which actions to take. At its most effective, data science is about decision intelligence—informing a decision making process to help achieve desired outcomes. These decisions come in many forms, from executive deliberation on strategic bets to streaming automations occurring millions of times per day.

Examples include:
* What price to charge for this ride share trip?
* Which product to recommend to this user?
* Should a school system reduce its class sizes?
* Where to deploy mobile health centers in an underserved rural area?

Behind each of these decisions is a set of what if questions of the form: What would happen if I did this? What would happen if I did that? To inform the decision we can predict answers to these what if questions. 

Often these answers are already valuable, but we can go one step further and recommend the action corresponding to the most favorable outcome. Thus our business problems can often be formulated as optimization problems of the form

$$ \underset{x}{\operatorname{argmax}} f(x) $$

where $x$ indexes the possible actions we can take and $f$ is a "what if" function that predicts the benefit of an action. The above expression says: "find the action that maximizes the predicted benefit."

So that's the setup; the key for optimal decision making is having a "what if" function that reliably maps actions to outcomes.

For data scientists of my generation who spent the early part of their careers getting good at machine learning, this probably  sounds an awful lot like a prediction problem.
We could follow the typical ML playbook , training a model that minimizes out of sample prediction error.
That means it would be the best predictor of  outcomes from actions  and an ideal objective function to use in our optimization task right?

Maybe; maybe not.

Unfortunately there are a few ways this can go right and  many ways this can go wrong. 
Using well-fitting statistical models or accuracy-maxing ML models for decision making can fail, not just slightly, but catastrophically. 
Many of these failure modes arise from errors in reasoning about what causes what and how, i.e. causality.
In this post we'll look at two types of prediction-decision failure modes that can occurr because of mishandling causality.
The first is about the importance of the causal mechanisms behind the correlations we exploit for prediction, and the second is about how these mechanisms can shift under our feet when taking a model from one environment to another.
The key idea is that using statistical or ML models for decision making requires careful causal reasoning, and lucky for you, this is the first post in a forthcoming series on Causal Inference.

So let's kick this series off by seeing some of the scary things that can happen when we naively use predictive models for decision making.

## Prediction-Decision Failure Modes

### Correlation does not imply causation — except when it does.

We've all had it drilled into our heads — 
*[correlation does not imply causation](https://en.wikipedia.org/wiki/Correlation_does_not_imply_causation)*  —
which refers to the inability to deduce a cause-and-effect relationship between two events  solely on the basis of an observed correlation or other statistical association between them.
This refrain is often acompanied by a [Tyler Vigen](https://www.tylervigen.com/spurious-correlations)-style example of very high correlation between two absurdly unrelated variables like the  popularity  of the surprised Pikachu meme and the number of middle school teachers in Puerto Rico.
But these "spurious correlations" between causally unrelated variables occur naturally by chance in finite samples, especially in very small samples like the ones in Tyler's examples.
They're just Type-I errors from running millions of hypothesis tests looking for non-zero correlation across pairs of unrelated variables; some pairs will happen to line up and show a high correlation simply by chance.



Reichenbach's Common Cause Principle states that if two events are correlated, there must either be a causal relationship between them or a third event that causes both. 





We know that predictive models work by leveraging statistical associations between a target variable and some predictor variables.





A predictive model exploits an association between a feature and a target.
Per Reichenbach, that association reflects some causal structure (barring chance and selection).
Chance aside (it fades with data), three structural explanations remain: A → B, B → A, or a common cause, possibly in combination.
The data alone can’t distinguish them, and the predictive model doesn’t need to.
A decision does need to, because only in the first does changing the feature change the target.
Cases follow.

### Wrong Causal Mechanism


The generalized essence is that association is not the effect of an action. A feature can predict an outcome because it causes it, because the outcome causes it, because both share a common cause, or because of selection. Prediction treats all four the same. A decision treats them very differently.

Suppose we're looking at some app useage data, and we want to figure out whether notifications can help to increase user engagement. 

* X: user enables push notifications during their first two weeks.
* Y: engagement, measured by the number of sessions in the first two weeks.

The data shows  a strong positive correlation between notifications and sessions in the data. 
We build a predictive model on this data, which picks up that correlation, predicting higher engagement when users have notifications enabled.
The decision: should we prompt every new user on day 1 to turn notifications on?

There are many different causal mechanisms that could lead to correlation between notifications and engagement.
Whether it's a good idea to nudge users to turn on notifications depends on which mechanism actually governs the system.

### 1. X causes Y

Notifications remind people to come back, so each enabled user opens the app more.

Structure: Notifications → Sessions
If you prompt everyone: Sessions go up. 

### 2. Y causes X (reverse causation)

Suppose the OS notification permission prompt is shown only after a user’s 5th session. 
Users with many sessions see the prompt and many of them accept, while users who open the app only a few times never see it. Notifications themselves do nothing for engagement.

Structure: Sessions → Sees prompt → Notifications on
Tell in the data: Almost nobody with fewer than 5 sessions has notifications on.

If you prompt everyone on day 1: You flip X without touching Y, and engagement doesn’t change.


### 3. Common cause (confounding)

Some users arrive through a friend’s invite (C). They turn notifications on to see what their friends are doing, and they also use the app more because of that social pull. The channel drives both, and notifications don’t affect engagement.

Structure: Notifications ← Invited by friend → Sessions
Tell in the data: Within each acquisition channel, notification-on and notification-off users look the same.

If you prompt everyone: Non-invited users enable notifications and nothing changes for them. Adjusting for channel would have revealed this.



## Failure to Generalize Across Environments

## Looking Ahead